# 🛍️ Shopping Mall Customer Segmentation
## Member 3 — DBSCAN Clustering (Optimized)
**Algorithm:** DBSCAN (Density-Based Spatial Clustering of Applications with Noise)

### 📌 老师反馈优化点：
1. **3D 可视化**：解决 2D PCA 图中点全部重叠（Overlapping）的问题，通过 3D 空间展示密度聚类的效果。
2. **V-measure Score**：添加外部评估指标，衡量聚类结果与真实标签（如性别）的一致性。
3. **详细解释**：解释 DBSCAN 的核心参数（eps, min_samples）以及为什么它能识别“噪声”。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, v_measure_score
from mpl_toolkits.mplot3d import Axes3D

import warnings
warnings.filterwarnings('ignore')

print('✅ Libraries imported successfully!')

## 1. Load Pre-processed Data

In [ ]:
X_scaled = np.load('X_scaled.npy')
df_clean = pd.read_csv('data/data_preprocessed.csv')
df_original = pd.read_csv('data/Shopping Mall Customer Segmentation Data .csv')

print(f'✅ Loaded X_scaled shape: {X_scaled.shape}')

## 2. Train DBSCAN Model
**意义**：DBSCAN 是一种基于密度的聚类算法。它不需要预先指定聚类数量，且能识别异常值（噪声）。
- `eps`: 邻域半径。如果半径太小，大部分点会被视为噪声；如果太大，所有点会聚成一类。
- `min_samples`: 成为核心点所需的最小邻居数。

In [ ]:
# 使用预设的较优参数（实际项目中可通过 K-distance 图确定）
dbscan = DBSCAN(eps=0.5, min_samples=5)
labels = dbscan.fit_predict(X_scaled)

df_clean['Cluster'] = labels
df_original['Cluster'] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = list(labels).count(-1)

print(f"Clusters found: {n_clusters}")
print(f"Noise points: {n_noise}")

## 3. Evaluation: V-measure Score
**意义**：衡量聚类结果与性别标签的一致性。

In [ ]:
v_score = v_measure_score(df_original['Gender'], labels)
print(f"V-measure Score: {v_score:.4f}")

print("\n💡 意义：DBSCAN 的 V-measure 通常反映了数据密度分布是否与已知类别（如性别）重合。")

## 4. 3D Cluster Visualization
**意义**：通过 3D 视角展示密度聚类的结果，灰色点代表被算法识别出的“噪声”或“异常值”。

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# 分别绘制噪声和簇
mask_noise = (labels == -1)
mask_clusters = (labels != -1)

# 噪声点用灰色表示
ax.scatter(df_original.loc[mask_noise, 'Age'], 
           df_original.loc[mask_noise, 'Annual Income'], 
           df_original.loc[mask_noise, 'Spending Score'], 
           c='lightgrey', label='Noise', s=10, alpha=0.3)

# 聚类点
scatter = ax.scatter(df_original.loc[mask_clusters, 'Age'], 
                     df_original.loc[mask_clusters, 'Annual Income'], 
                     df_original.loc[mask_clusters, 'Spending Score'], 
                     c=labels[mask_clusters], cmap='tab10', s=20, alpha=0.6)

ax.set_xlabel('Age')
ax.set_ylabel('Annual Income')
ax.set_zlabel('Spending Score')
ax.set_title('3D DBSCAN Clustering')
ax.legend()

plt.show()

print("💡 意义：在 3D 空间中，我们可以看到 DBSCAN 如何根据点的密度来划分区域。那些远离高密度区域的点被标记为噪声，这在 2D 图中很难区分。")